In [3]:
# 다중 선형 회귀(Multiple Linear Regression)
# 공부시간(x1), 학원 횟수(x2)를 사용해서 성적(y)을 예측

# 예측식
# y = w1*x1 + w2*x2 + b

# 학습 목표:
# 손실(loss)을 최소화하는 w1, w2, b 파라미터를 찾는다.

# 옵티마이저:
# 경사하강법(SGD)을 사용하여 파라미터를 업데이트

# 파라미터가 여러 개이므로
# 각 파라미터(w1, w2, b)에 대한 손실 함수의 편미분 값을 각각 계산해야 한다.
# gradient = 손실 함수(loss)를 각 파라미터 기준으로 미분한 값
# 손실함수를 w1 기준으로 편미분 => w1 기준 loss 변화율
# 손실함수를 w2 기준으로 편미분 => w2 기준 loss 변화율
# 손실함수를 b 기준으로 편미분 => b 기준 loss 변화율
# 즉,
# dL/dw1, dL/dw2, dL/db를(각각의 파라미터 gradient) 구한 뒤
# 경사하강법으로 모든 파라미터를 동시에 업데이트한다.

#각 파라미터마다 gradient를 각각 계산하고,
#optimizer가 그 각각의 gradient를 사용해서
#각 파라미터를 각각 업데이트한다.
#optimizer.apply_gradients(zip(gradients, variables)
# (dL_dw1, w1)
# (dL_dw2, w2)
# (dL_db, b)
# 이렇게 매칭해서 각 파라미터마다 gradient 계산
# -> 설정한 옵티마이저 알고리즘으로 각각 업데이트
# 실제 딥러닝에서는 GPU 병렬 연산을 통해
# 수많은 파라미터를 동시에 처리한다.

import tensorflow as tf
import numpy as np
#데이터
data = [[2,0,81], [4,4,93],[6,2,91],[8,3,97]]
#x1,x2,y 데이터 분리 - 원래 tf.constant()명시적 tensor변환해야함
x1 = [x_row1[0] for x_row1 in data]#[2,4,6,8] 공부시간 
x2 = [x_row2[1] for x_row2 in data]#[0,4,2,3] 학원횟수
y_data = [y_row[2] for y_row in data]#[81,93,91,97] 성적

#weight1, weight2, bias - 처음엔 랜덤으로 시작
w1 = tf.Variable(tf.random.uniform([1],0,10,dtype=tf.float64,seed=0))
w2 = tf.Variable(tf.random.uniform([1],0,10,dtype=tf.float64,seed=0))
b = tf.Variable(tf.random.uniform([1],0,100, dtype=tf.float64,seed=0))
#tf.constant: 고정값, 상수, 기본적으로 gradient 계산 대상 아님
# Variable:
# 학습 가능한 변수.
#GradientTape는 어떤 tensor를 기준으로 미분할지 추적
#GradientTape는 기본적으로 trainable=True인 
# tf.Variable을 자동 추적(기본값 true로 되어있음)
# GradientTape가 연산을 추적하여 gradient를 계산하고,
# optimizer(optimizer.apply_gradients()로)가 
# gradient를 사용해 값을 업데이트한다.

#예측값 y = w1*x1+w2*x2+b
def hypothesis(w1,w2,b):
    return  w1 * x1 + w2 * x2 + b
# w1,w2,b가 텐서라서 x1,x2 자동으로 텐서로 변환
# 백터 연산됨

#손실함수(RMSE):모델이 얼마나 틀렸는지 계산
def cost(w1,w2,b):
    return tf.sqrt(tf.reduce_mean(tf.square(hypothesis(w1,w2,b) - y_data)))

#옵티마이저
opt = tf.keras.optimizers.SGD(learning_rate=0.1)

#Training loop
for i in range(5000):
    with tf.GradientTape() as tape:
        current_cost = cost(w1,w2,b)#현재 손실
    grads = tape.gradient(current_cost,[w1,w2,b])#w1,w2,b기준으로 각각각 loss 편미분
    opt.apply_gradients(zip(grads, [w1,w2,b]))#옵티마이저로 각 파라미터 업데이트
    if i % 100 == 0:
        print(i,f'{current_cost.numpy()}, {w1.numpy()}, {w2.numpy()}, {b.numpy()}')
#Forward 순전파 : 예측값 계산
#Backpropagation 역전파: 각 파라미터와 loss의 관계에서 gradient 계산
#optimizer: 각 파라미터 업데이트


#w1 : 1.2
#w2 : 2.1
#b : 77.8 수렴


# 2400 1.837006049686887, [1.23010092], [2.16326687], [77.81167569]
# ...
# 4600 1.8370060496868776, [1.23010092], [2.16326687], [77.81167569]
# 4700 1.8370060496868776, [1.23010092], [2.16326687], [77.81167569]
# 4800 1.8370060496868776, [1.23010092], [2.16326687], [77.81167569]
# 4900 1.8370060496868776, [1.23010092], [2.16326687], [77.81167569]

0 21.416865139696796, [4.96852536], [0.50657417], [81.05836373]
100 1.8365511787115216, [1.0864067], [2.11383881], [78.85141816]
200 1.8369294629394104, [1.16784211], [2.14190843], [78.24812732]
300 1.836992642003952, [1.20363454], [2.154153], [77.99497574]
400 1.8370036867476807, [1.21892749], [2.15941323], [77.88867155]
500 1.8370056327598945, [1.22539724], [2.16164355], [77.84402041]
600 1.8370059761073423, [1.22812317], [2.16258413], [77.82526359]
700 1.8370060367010526, [1.22926976], [2.16297991], [77.81738399]
800 1.8370060473950434, [1.22975169], [2.16314629], [77.81407377]
900 1.8370060492824003, [1.2299542], [2.16321621], [77.81268313]
1000 1.837006049615482, [1.23003928], [2.16324558], [77.81209892]
1100 1.8370060496742875, [1.23007503], [2.16325792], [77.81185349]
1200 1.8370060496846565, [1.23009004], [2.16326311], [77.81175038]
1300 1.837006049686505, [1.23009635], [2.16326529], [77.81170707]
1400 1.8370060496868204, [1.230099], [2.1632662], [77.81168887]
1500 1.8370060496

In [ ]:
#(추론) 새로운 Data넣었을 때, 예측 확인
# 공부시간 3시간, 학원 횟수 2번
# 공부시간 9시간, 학원 횟수 5번
# 일 때, 점수 예측해보기
import tensorflow as tf
data = [[3,2],[9,5]]
new_x1 = tf.constant([x1_row[0] for x1_row in data], dtype=tf.float64)#[3,9] 공부시간
new_x2 = tf.constant([x2_row[1] for x2_row in data], dtype=tf.float64)#[2,5] 학원횟수

#예측값 y = w1 * x1 + w2 * x2 + b
pred_score = new_x1 * w1 + new_x2 * w2 + b
for hrs, acade_num, pred in zip(new_x1.numpy(), new_x2.numpy(), pred_score.numpy()):
    print(f'{hrs:.0f}시간 공부, {acade_num:.0f}번 학원 횟수 예상 점수 -> {pred:.2f}')

# 3시간 공부, 2번 학원 횟수 예상 점수 -> 88.03
# 9시간 공부, 5번 학원 횟수 예상 점수 -> 105.87

3시간 공부, 2번 학원 횟수 예상 점수 -> 88.03
9시간 공부, 5번 학원 횟수 예상 점수 -> 105.87


In [ ]:
# 최대 점수는 100점이라서 제한 주어야 함
new_x1 = tf.constant([x1_row[0] for x1_row in data], dtype=tf.float64)#[3,9] 공부시간
new_x2 = tf.constant([x2_row[1] for x2_row in data], dtype=tf.float64)#[2,5] 학원횟수

raw_scores = new_x1 * w1 + new_x2 * w2 + b #모델 예측값
pred_score = tf.clip_by_value(raw_scores,0.,100.)#0~100범위 제한
#tf.clip_by_value(tensor,min,max)
# 0보다 작으면 0으로 바꾸고, 100보다 크면 100으로 바꿈

for hrs, academy, pred in zip(new_x1.numpy(), new_x2.numpy(), pred_score.numpy()):
    print(f'{hrs:.0f}시간 공부, {academy:.0f}개 학원 예상 점수 -> {pred:.2f}')

# 3시간 공부, 2개 학원 예상 점수 -> 88.03
# 9시간 공부, 5개 학원 예상 점수 -> 100.00

3시간 공부, 2개 학원 예상 점수 -> 88.03
9시간 공부, 5개 학원 예상 점수 -> 100.00
